# 19 Python intermediate - `dataclasses` v2.0

_Kamil Bartocha_

## Rozkład jazdy

1. 🔹 **`@dataclass` i `field()`** - dekorator, `default_factory`, `repr`, `compare`
2. 🔹 **`__post_init__`, `ClassVar`, `InitVar`** - walidacja, pola klasowe, pola inicjalizacyjne
3. 🔹 **Zaawansowane wzorce** - `frozen`, dziedziczenie, `__slots__`, porównanie z `NamedTuple`

🐍 Każda sekcja zawiera ćwiczenia.

---

## 1. 🔹 `@dataclass` i `field()`

`@dataclass` automatycznie generuje `__init__`, `__repr__` i `__eq__`
na podstawie annotowanych pól. `field()` daje precyzyjną kontrolę
nad każdym polem.

| Parametr `field()` | Opis |
|---|---|
| `default` | wartość domyślna (dla typów niezmiennych) |
| `default_factory` | funkcja zwracająca wartość domyślną (dla list, dict) |
| `repr` | czy pole pojawia sie w `__repr__` (domyslnie True) |
| `compare` | czy pole jest uzywane w `__eq__` i `__lt__` (domyslnie True) |
| `hash` | czy pole jest uwzglednianie w `__hash__` |
| `init` | czy pole jest parametrem `__init__` (domyslnie True) |

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class Task:
    title:       str
    priority:    int = 1
    tags:        list[str] = field(default_factory=list)
    created_at:  datetime  = field(default_factory=datetime.now, repr=False)
    _id:         int        = field(default=0, init=False, repr=False)

    def add_tag(self, tag: str) -> None:
        self.tags.append(tag)


t1 = Task('Write tests', priority=3)
t2 = Task('Fix bug', tags=['urgent', 'backend'])

t1.add_tag('python')
print(t1)           # Task(title='Write tests', priority=3, tags=['python'])
print(t1 == t2)     # False
print(t1.created_at)

In [ ]:
from dataclasses import dataclass, field

# UWAGA: nie mozna uzyc mutowalnych wartosci domyslnych bezposrednio
# To zgłosi blad:
# @dataclass
# class Bad:
#     items: list = []   # ValueError!

# Poprawnie - uzyj field(default_factory=list)
@dataclass
class Config:
    name:    str
    options: dict = field(default_factory=dict)
    flags:   list = field(default_factory=list)

c = Config('main')
c.options['debug'] = True
print(c)

---

### 🐍 Ćwiczenia - @dataclass i field()

**Ćw. 1.** Napisz `@dataclass` dla klasy `Product` z polami:
`name: str`, `price: float`, `stock: int = 0`,
`tags: list[str]` (pusta lista domyslnie).
Dodaj metodę `is_available()` zwracającą `stock > 0`.

**Ćw. 2.** Napisz `@dataclass` dla klasy `Vector2D` z polami
`x: float` i `y: float`. Dodaj metody: `__add__`, `__mul__(scalar)`,
`length()`. Uzyj `field(compare=False)` dla pola `length_cached`
obliczanego przy tworzeniu.

**Ćw. 3. *(Trudniejsze)*** Napisz `@dataclass(order=True)` dla klasy
`Student` z polami `name: str`, `grade: float`, `student_id: int`.
Uzyj `field(compare=False)` dla `name` i `student_id` - sortowanie
ma byc tylko po `grade`. Posortuj listę studentów.

In [ ]:
# Ćw. 1
from dataclasses import dataclass, field

@dataclass
class Product:
    name:  str
    price: float
    stock: int      = ...
    tags:  list[str] = field(...)

    def is_available(self) -> bool:
        ...


p = Product('Laptop', 2999.0, stock=5, tags=['electronics'])
print(p)
print(p.is_available())   # True

In [ ]:
# Ćw. 2
from dataclasses import dataclass, field
import math

@dataclass
class Vector2D:
    x: float
    y: float
    length_cached: float = field(init=False, compare=False)

    def __post_init__(self):
        self.length_cached = math.sqrt(self.x**2 + self.y**2)

    def __add__(self, other: 'Vector2D') -> 'Vector2D':
        ...

    def __mul__(self, scalar: float) -> 'Vector2D':
        ...

    def length(self) -> float:
        return self.length_cached


v1 = Vector2D(3.0, 4.0)
v2 = Vector2D(1.0, 2.0)
print(v1.length())          # 5.0
print((v1 + v2).x)          # 4.0
print((v1 * 2).x)           # 6.0

In [ ]:
# Ćw. 3
from dataclasses import dataclass, field

@dataclass(order=True)
class Student:
    grade:      float
    name:       str   = field(compare=False)
    student_id: int   = field(compare=False)


students = [
    Student(name='Alice', grade=4.5, student_id=1),
    Student(name='Bob',   grade=3.0, student_id=2),
    Student(name='Carol', grade=4.5, student_id=3),
    Student(name='Dave',  grade=2.0, student_id=4),
]

for s in sorted(students):
    print(f'{s.name}: {s.grade}')

---

## 2. 🔹 `__post_init__`, `ClassVar`, `InitVar`

- `__post_init__` - wywoływany po `__init__`, uzywamy do walidacji
  i obliczania pól pochodnych
- `ClassVar[T]` - pole klasowe (nie instancji), pomijane przez `__init__`
- `InitVar[T]` - parametr `__init__` ktory **nie staje sie polem**;
  przekazywany do `__post_init__`

In [ ]:
from dataclasses import dataclass, field, InitVar
from typing import ClassVar

@dataclass
class Circle:
    radius: float
    # ClassVar - wspoldzielone przez wszystkie instancje, nie w __init__
    PI: ClassVar[float] = 3.14159
    # InitVar - tylko parametr __init__, nie staje sie polem
    validate: InitVar[bool] = True

    area:        float = field(init=False)
    perimeter:   float = field(init=False)

    def __post_init__(self, validate: bool):
        if validate and self.radius <= 0:
            raise ValueError(f'Promien musi byc dodatni, podano: {self.radius}')
        self.area      = Circle.PI * self.radius ** 2
        self.perimeter = 2 * Circle.PI * self.radius


c = Circle(5.0)
print(c)              # Circle(radius=5.0, area=78.53..., perimeter=31.41...)
print(Circle.PI)      # 3.14159

try:
    Circle(-1.0)
except ValueError as e:
    print(e)

---

### 🐍 Ćwiczenia - __post_init__, ClassVar, InitVar

**Ćw. 4.** Napisz `@dataclass` dla klasy `Rectangle` z polami
`width: float` i `height: float`. Uzyj `__post_init__` do walidacji
(oba > 0) i obliczenia `area` i `perimeter` jako pól pochodnych.

**Ćw. 5.** Napisz `@dataclass` dla klasy `User` z polami:
`first_name: str`, `last_name: str`, `email: str`.
Dodaj `ClassVar[int]` licznik instancji `count`. Zwiększaj go
w `__post_init__`. Dodaj pole pochodne `full_name: str`.

**Ćw. 6. *(Trudniejsze)*** Napisz `@dataclass` dla klasy `Temperature`.
`InitVar` `unit: str` ('C', 'F', 'K') okresla w jakiej jednostce
podawana jest wartosc `value`. `__post_init__` przelicza wszystko
na Celsjusze. Dodaj metody `to_fahrenheit()` i `to_kelvin()`.

In [ ]:
# Ćw. 4
from dataclasses import dataclass, field

@dataclass
class Rectangle:
    width:     float
    height:    float
    area:      float = field(init=False, repr=False)
    perimeter: float = field(init=False, repr=False)

    def __post_init__(self):
        ...


r = Rectangle(4.0, 5.0)
print(r)            # Rectangle(width=4.0, height=5.0)
print(r.area)       # 20.0
print(r.perimeter)  # 18.0

In [ ]:
# Ćw. 5
from dataclasses import dataclass, field
from typing import ClassVar

@dataclass
class User:
    first_name: str
    last_name:  str
    email:      str
    count:      ClassVar[int] = 0
    full_name:  str = field(init=False)

    def __post_init__(self):
        ...


u1 = User('Alice', 'Smith', 'alice@example.com')
u2 = User('Bob',   'Jones', 'bob@example.com')
print(u1.full_name)   # Alice Smith
print(User.count)     # 2

In [ ]:
# Ćw. 6
from dataclasses import dataclass, field, InitVar

@dataclass
class Temperature:
    value:   float
    unit:    InitVar[str] = 'C'
    celsius: float = field(init=False)

    def __post_init__(self, unit: str):
        # hint: przelicz value na celsius w zaleznosci od unit
        ...

    def to_fahrenheit(self) -> float:
        return self.celsius * 9/5 + 32

    def to_kelvin(self) -> float:
        return self.celsius + 273.15


t_c = Temperature(100, 'C')
t_f = Temperature(212, 'F')
t_k = Temperature(373.15, 'K')
print(t_c.celsius)   # 100.0
print(t_f.celsius)   # 100.0
print(t_k.celsius)   # 100.0

---

## 3. 🔹 Zaawansowane wzorce

| Opcja | Opis |
|---|---|
| `@dataclass(frozen=True)` | niemutowalny, generuje `__hash__` |
| `@dataclass(slots=True)` | uzywa `__slots__`, szybszy dostep, mniej pamieci (Python 3.10+) |
| `@dataclass(order=True)` | generuje `__lt__`, `__le__`, `__gt__`, `__ge__` |
| `dataclasses.asdict(obj)` | konwersja do slownika |
| `dataclasses.astuple(obj)` | konwersja do krotki |
| `dataclasses.replace(obj, **changes)` | kopia z modyfikacjami |

In [ ]:
from dataclasses import dataclass, field, asdict, astuple, replace

@dataclass(frozen=True)
class Point:
    x: float
    y: float


p1 = Point(1.0, 2.0)
p2 = Point(1.0, 2.0)
print(p1 == p2)   # True
print(hash(p1))   # mozna uzyc jako klucz dict

# Modyfikacja przez replace (zwraca nowy obiekt)
p3 = replace(p1, x=10.0)
print(p3)   # Point(x=10.0, y=2.0)
print(p1)   # Point(x=1.0, y=2.0) - niezmieniony

# Konwersja
print(asdict(p1))    # {'x': 1.0, 'y': 2.0}
print(astuple(p1))   # (1.0, 2.0)

---

### 🐍 Ćwiczenia - zaawansowane wzorce

**Ćw. 7.** Napisz `@dataclass(frozen=True)` dla klasy `Color`
z polami `r: int`, `g: int`, `b: int` (0-255). Uzyj `__post_init__`
do walidacji zakresu. Sprawdź ze mozna uzywac jako klucz slownika.

**Ćw. 8.** Napisz hierarchię klas `Shape` (bazowa, frozen) i
`Circle(Shape)` oraz `Square(Shape)` (dziedziczące). Uzyj `asdict()`
i `replace()` w przykładach użycia.

**Ćw. 9. *(Trudniejsze)*** Porównaj `@dataclass`, `NamedTuple`
i `TypedDict` - napisz to samo `Point3D(x, y, z)` na trzy sposoby
i sprawdź: mutowalność, hashability, dostep przez indeks,
zuzycie pamieci (`sys.getsizeof`), czy dziedziczy po `tuple`.

In [ ]:
# Ćw. 7
from dataclasses import dataclass

@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int

    def __post_init__(self):
        for name, val in [('r', self.r), ('g', self.g), ('b', self.b)]:
            if not 0 <= val <= 255:
                raise ValueError(f'{name}={val} poza zakresem 0-255')


red = Color(255, 0, 0)
palette = {red: 'red', Color(0, 255, 0): 'green'}
print(palette[red])   # red
print(palette[Color(255, 0, 0)])   # red

In [ ]:
# Ćw. 8
from dataclasses import dataclass, field, asdict, replace
import math

@dataclass(frozen=True)
class Shape:
    color: str = 'black'

    def area(self) -> float:
        raise NotImplementedError

@dataclass(frozen=True)
class Circle(Shape):
    radius: float = 1.0

    def area(self) -> float:
        return math.pi * self.radius ** 2

@dataclass(frozen=True)
class Square(Shape):
    side: float = 1.0

    def area(self) -> float:
        return self.side ** 2


c = Circle(color='red', radius=5.0)
print(asdict(c))
bigger = replace(c, radius=10.0)
print(bigger.area())

In [ ]:
# Ćw. 9
from dataclasses import dataclass
from typing import NamedTuple, TypedDict
import sys

# 1. dataclass
@dataclass
class Point3D_DC:
    x: float; y: float; z: float

# 2. NamedTuple
class Point3D_NT(NamedTuple):
    x: float; y: float; z: float

# 3. TypedDict
class Point3D_TD(TypedDict):
    x: float; y: float; z: float


dc = Point3D_DC(1.0, 2.0, 3.0)
nt = Point3D_NT(1.0, 2.0, 3.0)
td = Point3D_TD(x=1.0, y=2.0, z=3.0)

print('--- mutowalnosc ---')
dc.x = 99        # OK
# nt.x = 99      # AttributeError
print('DC mutowalny, NT niemutowalny')

print('--- hashability ---')
print(hash(nt))  # OK dla NamedTuple
try: hash(dc)
except TypeError: print('DC nie jest hashowalny (domyslnie)')

print('--- dostep przez indeks ---')
print(nt[0])     # 1.0
try: dc[0]
except TypeError: print('DC nie obsługuje indeksowania')

print('--- rozmiar w pamieci ---')
print(f'DC:  {sys.getsizeof(dc)} B')
print(f'NT:  {sys.getsizeof(nt)} B')
print(f'TD:  {sys.getsizeof(td)} B')

print('--- dziedziczenie po tuple ---')
print(isinstance(nt, tuple))   # True
print(isinstance(dc, tuple))   # False